<a href="https://colab.research.google.com/github/kieshaaaa/aircraft-enginer-rul-prediction/blob/main/classical_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. first we load our CMAPP dataset. we first make use of dataset FD001

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

columns = (
    ["engine_id", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train = pd.read_csv("/content/train_FD001.txt", sep=r"\s+", header=None, names=columns)
test = pd.read_csv("/content/test_FD001.txt", sep=r"\s+", header=None, names=columns)
test_rul = pd.read_csv("/content/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

train.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/train_FD001.txt'

2. Now we check the data structure

In [ ]:
print("Training shape:", train.shape)
print("Test shape:", test.shape)
print("Test RUL shape:", test_rul.shape)
print("Training engines:", train["engine_id"].nunique())
print("Test engines:", test["engine_id"].nunique())
print("Missing training values:", train.isna().sum().sum())
print("Missing test values:", test.isna().sum().sum())

lifetimes = train.groupby("engine_id")["cycle"].max()
print("\nTraining engine lifetimes (cycles):")
print("Shortest:", lifetimes.min(), "| Longest:", lifetimes.max(), "| Average:", round(lifetimes.mean(), 1))

3. now we calculate the RUL from the training data

In [ ]:
final_cycle = train.groupby("engine_id")["cycle"].transform("max")
train["RUL"] = final_cycle - train["cycle"]

print(train[["engine_id", "cycle", "RUL"]].head(10))

4. we cap RUL at 125

In [ ]:
train["RUL_capped"] = train["RUL"].clip(upper=125)

print(train[["engine_id", "cycle", "RUL", "RUL_capped"]].head(10))
print("Max capped RUL:", train["RUL_capped"].max())

5. we plot sensor readings for one engine

In [ ]:
engine = train[train["engine_id"] == 1]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, sensor in zip(axes.flat, ["sensor_2", "sensor_4", "sensor_7", "sensor_11"]):
    ax.plot(engine["cycle"], engine[sensor])
    ax.set_title(f"Engine 1: {sensor}")
    ax.set_xlabel("Operating cycle")
    ax.grid(True)

plt.tight_layout()
plt.show()

6. now  find and drop uninformative sensors

In [ ]:
sensor_cols = [f"sensor_{i}" for i in range(1, 22)]

print("Standard deviation of each sensor:")
print(train[sensor_cols].std().sort_values())

drop_sensors = [c for c in sensor_cols if train[c].std() < 0.01]
features = [c for c in sensor_cols if c not in drop_sensors]

print("\nDropped:", drop_sensors)
print("Using", len(features), "sensors:", features)

7. split the training engine 80/20

In [ ]:
rng = np.random.default_rng(42)
engines = train["engine_id"].unique()
val_engines = rng.choice(engines, size=20, replace=False)

tr = train[~train["engine_id"].isin(val_engines)]
val = train[train["engine_id"].isin(val_engines)]

print("Training engines:", tr["engine_id"].nunique(), "| rows:", len(tr))
print("Validation engines:", val["engine_id"].nunique(), "| rows:", len(val))
print("Overlap between the two:", set(tr["engine_id"]) & set(val["engine_id"]))

8. scaling the sensors,fitted on training engines only

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_tr = scaler.fit_transform(tr[features])
y_tr = tr["RUL_capped"].values

X_val = scaler.transform(val[features])
y_val = val["RUL_capped"].values

print("X_tr:", X_tr.shape, "| X_val:", X_val.shape)
print("Training min/max after scaling:", X_tr.min().round(2), X_tr.max().round(2))

9. Preparing the test set (last cycle of each engine)

In [ ]:
test_last = test.groupby("engine_id").last().reset_index()
X_test = scaler.transform(test_last[features])
y_test = test_rul["RUL"].values

print("X_test:", X_test.shape, "| y_test:", y_test.shape)

NameError: name 'test' is not defined

10: Evaluation function

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

results = []

def evaluate(name, model):
    row = {"Model": name}
    for split, X, y in [("Val", X_val, y_val), ("Test", X_test, y_test)]:
        pred = np.clip(model.predict(X), 0, None)
        row[f"{split} MAE"] = round(mean_absolute_error(y, pred), 2)
        row[f"{split} RMSE"] = round(np.sqrt(mean_squared_error(y, pred)), 2)
    results.append(row)
    print(row)

11. naive baseline and linear regression

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

baseline = DummyRegressor(strategy="mean").fit(X_tr, y_tr)
evaluate("Naive baseline (mean RUL)", baseline)

lr = LinearRegression().fit(X_tr, y_tr)
evaluate("Linear Regression", lr)

NameError: name 'X_tr' is not defined

12. Random forest and sensor importance

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)
evaluate("Random Forest", rf)

importance = pd.Series(rf.feature_importances_, index=features).sort_values()
importance.plot(kind="barh", figsize=(8, 5), title="Random Forest: sensor importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

13. results table

In [ ]:
pd.DataFrame(results)

14. adding the nasa score and re-scoring the models based on that


In [ ]:
def nasa_score(y_true, y_pred):
    d = y_pred - y_true   # positive = late prediction (dangerous)
    return np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1))

results = []

def evaluate(name, model):
    row = {"Model": name}
    for split, X, y in [("Val", X_val, y_val), ("Test", X_test, y_test)]:
        pred = np.clip(model.predict(X), 0, None)
        row[f"{split} MAE"] = round(mean_absolute_error(y, pred), 2)
        row[f"{split} RMSE"] = round(np.sqrt(mean_squared_error(y, pred)), 2)
    test_pred = np.clip(model.predict(X_test), 0, None)
    row["Test NASA score"] = round(nasa_score(y_test, test_pred), 1)
    results.append(row)
    print(row)

for name, model in [("Naive baseline (mean RUL)", baseline),
                    ("Linear Regression", lr),
                    ("Random Forest", rf)]:
    evaluate(name, model)